# Small Business Profitability Analysis
**Project ID:** DA-019  
**Domain:** Business Analytics  

### Objective:
Analyze sales, costs, discounts, and product categories to understand profitability patterns and identify which products or periods generate stronger profit margins.


### 1. Import Libraries


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8


### 2. Load and Inspect Dataset
We load the retail transactions dataset (`small_business_sales.csv`) which contains sales, product pricing, discount rates, and order statuses.


In [ ]:
# Load dataset
data_path = os.path.join("..", "data", "small_business_sales.csv")
df_raw = pd.read_csv(data_path)

print(f"Dataset Shape: {df_raw.shape}")
df_raw.head()


In [ ]:
# Check column types and missing values
df_raw.info()


### 3. Data Cleaning & Feature Extraction
- Check for duplicates and handle missing values.
- Convert date column to datetime and extract month and quarter.
- Categorize discount percentages into standard tiers for easier analysis.


In [ ]:
df = df_raw.copy()

# Remove any duplicate transactions
df = df.drop_duplicates(subset=['Transaction_ID'])

# Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Quarter'] = 'Q' + df['Date'].dt.quarter.astype(str)

# Create discount buckets
bins = [-1, 0, 10, 20, 30, 100]
labels = ['No Discount (0%)', 'Low (1-10%)', 'Moderate (11-20%)', 'High (21-30%)', 'Deep (>30%)']
df['Discount_Bracket'] = pd.cut(df['Discount_Percent'], bins=bins, labels=labels)

df[['Transaction_ID', 'Date', 'Month_Name', 'Discount_Bracket', 'Return_Status']].head()


### 4. Financial Calculations
> **Note:** Revenue is not profit. Profit is calculated after deducting product cost (COGS) and accounting for discounts and returns.

**Formulas used:**
1. Gross Revenue = $\text{Unit Price} \times \text{Units Sold}$
2. Discount Amount = $\text{Gross Revenue} \times (\text{Discount Percent} / 100)$
3. Net Revenue = $\text{Gross Revenue} - \text{Discount Amount}$ (0 if returned)
4. Total COGS = $\text{Unit Cost} \times \text{Units Sold}$ (0 if returned and restocked)
5. Net Profit = $\text{Net Revenue} - \text{Total COGS}$
6. Profit Margin (%) = $(\text{Net Profit} / \text{Net Revenue}) \times 100$


In [ ]:
# Calculate calculated columns
df['Gross_Revenue'] = (df['Unit_Price'] * df['Units_Sold']).round(2)
df['Discount_Amount'] = (df['Gross_Revenue'] * (df['Discount_Percent'] / 100.0)).round(2)

# If an order is returned, Net Revenue is 0
df['Net_Revenue'] = np.where(
    df['Return_Status'] == 'Yes',
    0.0,
    df['Gross_Revenue'] - df['Discount_Amount']
).round(2)

# COGS is 0 for returned orders because goods are restocked
df['Total_COGS'] = np.where(
    df['Return_Status'] == 'Yes',
    0.0,
    df['Unit_Cost'] * df['Units_Sold']
).round(2)

# Net Profit
df['Net_Profit'] = (df['Net_Revenue'] - df['Total_COGS']).round(2)

# Profit Margin %
df['Profit_Margin_Pct'] = np.where(
    df['Net_Revenue'] > 0,
    (df['Net_Profit'] / df['Net_Revenue']) * 100.0,
    0.0
).round(2)

df[['Transaction_ID', 'Unit_Price', 'Units_Sold', 'Gross_Revenue', 'Net_Revenue', 'Total_COGS', 'Net_Profit', 'Profit_Margin_Pct']].head()


In [ ]:
# Overall Summary
valid_sales = df[df['Return_Status'] == 'No']

summary = pd.DataFrame({
    'Metric': [
        'Total Transactions',
        'Successful Sales',
        'Returned Orders',
        'Total Net Revenue (₹)',
        'Total COGS (₹)',
        'Total Net Profit (₹)',
        'Overall Profit Margin (%)'
    ],
    'Value': [
        f"{len(df):,}",
        f"{len(valid_sales):,}",
        f"{(df['Return_Status'] == 'Yes').sum()}",
        f"₹{valid_sales['Net_Revenue'].sum():,.2f}",
        f"₹{valid_sales['Total_COGS'].sum():,.2f}",
        f"₹{valid_sales['Net_Profit'].sum():,.2f}",
        f"{(valid_sales['Net_Profit'].sum() / valid_sales['Net_Revenue'].sum()) * 100:.2f}%"
    ]
})

summary


### 5. Category-wise Profitability Comparison
Comparing performance across categories to see if higher revenue leads to higher profit.
